In [ ]:
# 1
import time
import requests
import csv
from bs4 import BeautifulSoup

URL = "https://www.hollys.co.kr/store/korea/korStore2.do"
HEADERS = {
    'User-Agent': 'Mozilla/5.0'
}
DELAY = 0.7
result = []

def fetch(page):
    params = {"pageNo": page, "sido": "", "gugun": "", "store": ""}
    r = requests.get(URL, params=params, headers=HEADERS, timeout=10)
    r.raise_for_status() 
    return r.text

def parse(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    
    trs = soup.select("table.tb_store tbody tr")
    
    for tr in trs:
        td = tr.select("td")
        
        if len(td) < 6:
            continue
    
        services = []
        for img in td[4].select("img"):
            if 'alt' in img.attrs:
                services.append(img.attrs['alt'].strip())
    
        rows.append({
            "지역": td[0].text.strip(),
            "매장명": td[1].text.strip(),
            "현황": td[2].text.strip(),
            "주소": td[3].text.strip(),
            "매장 서비스": services,
            "전화번호": td[5].text.strip(),
        })
        
    return rows

for page in range(1, 11):
    rows = parse(fetch(page))
    
    if not rows: 
        print(f"{page}페이지가 비어 있습니다.")
        break
        
    result.extend(rows)
    print(f"{page}페이지 수집 완료: 누적 {len(result)}건")
    
    time.sleep(DELAY)

pd.DataFrame(result).to_csv("hollys.csv", index=False, encoding="utf-8-sig")

result[:1]

1페이지 수집 완료: 누적 10건
2페이지 수집 완료: 누적 20건
3페이지 수집 완료: 누적 30건
4페이지 수집 완료: 누적 40건
5페이지 수집 완료: 누적 50건
6페이지 수집 완료: 누적 60건
7페이지 수집 완료: 누적 70건
8페이지 수집 완료: 누적 80건
9페이지 수집 완료: 누적 90건
10페이지 수집 완료: 누적 100건


[{'지역': '대전 유성구',
  '매장명': '대전도안마을점',
  '현황': '영업중',
  '주소': '대전광역시 유성구 도안대로 560 (도안마을1단지) /봉명동 1024',
  '매장 서비스': [],
  '전화번호': '042-826-6080'}]

In [ ]:
#2
import time
import requests
import pandas as pd  
from bs4 import BeautifulSoup

URL = "https://www.aladin.co.kr/shop/common/wbest.aspx"
HEADERS = {
    'User-Agent': 'Mozilla/5.0'
}
DELAY = 2
result = []

def fetch(page):
    params = {
        "BranchType": 1,
        "page": page
    }
    r = requests.get(URL, params=params, headers=HEADERS, timeout=10)
    r.raise_for_status() 
    return r.text

def parse(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    
    book_boxes = soup.select("div.ss_book_box")
    
    for box in book_boxes:
        category_tag = box.select_one("li a[href*='CID']")
        category = category_tag.text.strip() if category_tag else "[국내도서]"
        
        title_tag = box.select_one("a.bo3 b")
        title = title_tag.text.strip() if title_tag else "제목 없음"
        
        author = ""
        for li in box.select("li"):
            if "|" in li.text:
                author = li.text.split("|")[0].strip()
                break
                
        price_regular_tag = box.select_one("span.ss_p1")
        price_regular = price_regular_tag.text.strip() if price_regular_tag else ""
        
        price_discount_tag = box.select_one("span.ss_p2")
        price_discount = price_discount_tag.text.strip() if price_discount_tag else ""
        
        img_tag = box.select_one("img.i_cover, img.front_cover")
        img_url = img_tag.attrs.get('src', '') if img_tag else ""
        
        rows.append({
            "카테고리": category,
            "제목": title,
            "저자": author,
            "정가": price_regular,
            "할인가격": price_discount,
        })
        
    return rows


for page in range(1, 11):
    html = fetch(page)
    rows = parse(html)
    
    if not rows: 
        print(f"{page}페이지에 데이터가 없습니다.")
        break
        
    result.extend(rows)
    print(f"{page}페이지 수집 완료: 누적 {len(result)}건")
    
    time.sleep(DELAY)

df = pd.DataFrame(result)
df.to_csv("aladin_bestseller.csv", index=False, encoding="utf-8-sig")

result[:2]


1페이지 수집 완료: 누적 50건
2페이지 수집 완료: 누적 100건
3페이지 수집 완료: 누적 150건
4페이지 수집 완료: 누적 200건
5페이지 수집 완료: 누적 250건
6페이지 수집 완료: 누적 300건
7페이지 수집 완료: 누적 350건
8페이지 수집 완료: 누적 400건
9페이지 수집 완료: 누적 450건
10페이지 수집 완료: 누적 500건


[{'카테고리': '[국내도서]',
  '제목': '제목 없음',
  '저자': '루키우스 안나이우스 세네카 (지은이), 하와이 대저택 (편역)',
  '정가': '',
  '할인가격': '16,200원',
  '이미지': 'https://image.aladin.co.kr/product/39640/49/cover200/k872130175_1.jpg'},
 {'카테고리': '[국내도서]',
  '제목': '제목 없음',
  '저자': '김애란 (지은이)',
  '정가': '',
  '할인가격': '15,300원',
  '이미지': 'https://image.aladin.co.kr/product/40019/36/cover200/k742130236_1.jpg'}]